In [1]:
import requests
import json
import pandas as pd
import re
import time
import sys
from bs4 import BeautifulSoup

### 릭스트림 언급만

In [ ]:
# 1. 본문 텍스트 추출 함수
def get_blog_text(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/110.0.0.0"}
    try:
        res = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(res.text, "html.parser")
        if soup.find("iframe", id="mainFrame"):
            real_url = "https://blog.naver.com" + soup.find("iframe", id="mainFrame")["src"]
            res_real = requests.get(real_url, headers=headers)
            soup_real = BeautifulSoup(res_real.text, "html.parser")
            content = soup_real.select_one(".se-main-container") or soup_real.select_one("#postViewArea")
            return content.get_text(" ", strip=True) if content else ""
    except: return ""
    return ""

# 2. 메인 수집 함수
def scrape_rextreme_all_pages():
    headers = {
        'X-Naver-Client-Id' : 'qlURw5tgE7CmccdW37fe', 
        'X-Naver-Client-Secret' : 'FOGoT6pEeT'
    }
    
    keywords = ["렉스트림", "rextreme"]
    total_items = []

    # [단계 1] 모든 페이지 목록 가져오기 (100건씩 끊어서 반복)
    print("🔎 검색 결과 목록을 전수 수집 중입니다...")
    for kw in keywords:
        # 네이버 API는 최대 1,000건까지 지원하므로 10번 반복 (100개씩)
        for i in range(10):
            start_pos = (i * 100) + 1
            url = f"https://openapi.naver.com/v1/search/blog.json?query={kw}&display=100&start={start_pos}&sort=date"
            res = requests.get(url, headers=headers)
            
            if res.status_code == 200:
                data = json.loads(res.text)
                items = data.get('items', [])
                if not items: break # 더 이상 가져올 데이터가 없으면 중단
                total_items.extend(items)
                
                # 전체 검색 결과 건수보다 많이 가져올 수 없으므로 체크
                if len(total_items) >= data.get('total', 0): break
            else:
                break
            time.sleep(0.1)

    # 중복 제거 (링크 기준)
    df = pd.DataFrame(total_items).drop_duplicates(subset=['link']).reset_index(drop=True)
    total_count = len(df)
    
    if total_count == 0:
        print("❌ 검색 결과가 없습니다.")
        return

    # [단계 2] 본문 수집 및 필터링
    print(f"🚀 총 {total_count}건의 블로그 분석 및 원문 저장을 시작합니다.")
    
    clean_results = []
    target_words = ['렉스트림', 'rextreme']
    exclude_words = ['주식회사', '인허가', '소재지', '면적', '취득일']

    for idx, row in df.iterrows():
        full_text = get_blog_text(row['link'])
        title = re.sub('<.*?>', '', row['title'])
        
        # 소문자로 변환하여 대소문자 상관없이 검사
        is_target = any(word in title.lower() or word in full_text.lower() for word in target_words)
        is_garbage = any(word in full_text for word in exclude_words)

        if is_target and not is_garbage:
            clean_results.append({
                'title': title,
                'postdate': row['postdate'],
                'fulltext': full_text,
                'link': row['link']
            })
        
        # 진행률 표시 (한 줄 업데이트)
        percent = ((idx + 1) / total_count) * 100
        sys.stdout.write(f"\r📊 진행도: {percent:6.2f}% | 처리중: {idx+1}/{total_count} | 저장 확정: {len(clean_results)}건")
        sys.stdout.flush()
        
        time.sleep(1.2)

    # [단계 3] 결과 저장
    final_df = pd.DataFrame(clean_results)
    final_df.to_csv('rextreme_naver_reviews.csv', index=False, encoding='utf-8-sig')
    print(f"\n\n✨ 완료! 총 {len(final_df)}건의 유효한 리뷰 데이터가 저장되었습니다.")

# 실행
scrape_rextreme_all_pages()

🔎 검색 결과 목록을 전수 수집 중입니다...
🚀 총 242건의 블로그 분석 및 원문 저장을 시작합니다.
📊 진행도: 100.00% | 처리중: 242/242 | 저장 확정: 136건

✨ 완료! 총 136건의 유효한 리뷰 데이터가 저장되었습니다.


### 릭스트림 + 세방전지 언급

In [9]:
import requests
import json
import pandas as pd
import re
import time
import sys
from bs4 import BeautifulSoup

# 1. 본문 텍스트 추출 함수 (iframe 구조 대응)
def get_blog_text(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/110.0.0.0"}
    try:
        res = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(res.text, "html.parser")
        if soup.find("iframe", id="mainFrame"):
            real_url = "https://blog.naver.com" + soup.find("iframe", id="mainFrame")["src"]
            res_real = requests.get(real_url, headers=headers)
            soup_real = BeautifulSoup(res_real.text, "html.parser")
            content = soup_real.select_one(".se-main-container") or soup_real.select_one("#postViewArea")
            return content.get_text(" ", strip=True) if content else ""
    except:
        return ""
    return ""

# 2. 그룹별 수집 함수
def collect_by_group(keywords, group_label, target_count=500):
    headers = {
        'X-Naver-Client-Id' : 'qlURw5tgE7CmccdW37fe', 
        'X-Naver-Client-Secret' : 'FOGoT6pEeT'
    }
    
    group_results = []
    unique_links = set() # 중복 수집 방지
    exclude_words = ['주식회사', '인허가', '소재지', '면적', '취득일']

    print(f"\n📂 [{group_label}] 그룹 수집 시작 (목표: {target_count}건, 정확도순)")

    for kw in keywords:
        if len(group_results) >= target_count: break
        
        # 정확도 순(sim)으로 최대 10페이지까지 탐색
        for i in range(10):
            if len(group_results) >= target_count: break
            
            start_pos = (i * 100) + 1
            url = f"https://openapi.naver.com/v1/search/blog.json?query={kw}&display=100&start={start_pos}&sort=sim"
            res = requests.get(url, headers=headers)
            
            if res.status_code == 200:
                data = json.loads(res.text)
                items = data.get('items', [])
                if not items: break

                for item in items:
                    if len(group_results) >= target_count: break
                    
                    link = item['link']
                    if link not in unique_links:
                        # 전문 수집
                        full_text = get_blog_text(link)
                        if not full_text: continue # 본문이 없으면 건너뜀
                        
                        title = re.sub('<.*?>', '', item['title'])
                        
                        # 필터링 로직 (소문자 변환 검사 및 제외 단어 체크)
                        is_target = any(word.lower() in title.lower() or word.lower() in full_text.lower() for word in keywords)
                        is_garbage = any(word in full_text for word in exclude_words)

                        if is_target and not is_garbage:
                            group_results.append({
                                'group': group_label, # 어떤 그룹인지 기록
                                'title': title,
                                'postdate': item['postdate'],
                                'fulltext': full_text,
                                'link': link
                            })
                            unique_links.add(link)
                            
                            # 실시간 진행률 표시
                            sys.stdout.write(f"\r📊 {group_label} 진행도: {len(group_results)}/{target_count} 건 완료")
                            sys.stdout.flush()
                            time.sleep(1.2) # 네이버 차단 방지
            else:
                print(f"\n❌ API 호출 에러: {res.status_code}")
                break
    
    return group_results

# 3. 메인 실행부
def main():
    # 키워드 그룹 정의
    rextreme_keywords = ['렉스트림', '랙스트림', 'REXTREME', 'rextreme']
    brand_keywords = ['세방전지', 'SEBANG', 'sebang', '로케트 배터리', '로케트배터리', 'ROCKET BATTERY', 'rocket battery', 'AGM']

    # 각 그룹 수집 (각각 500건 목표)
    rextreme_list = collect_by_group(rextreme_keywords, 'rextreme', 500)
    brand_list = collect_by_group(brand_keywords, 'brand', 500)

    # 데이터 통합
    final_df = pd.DataFrame(rextreme_list + brand_list)

    if not final_df.empty:
        # 결과 저장 (UTF-8-SIG로 저장해야 엑셀에서 한글이 안 깨집니다)
        final_df.to_csv('combined_brand_reviews.csv', index=False, encoding='utf-8-sig')
        print(f"\n\n✨ 수집 완료! 최종 {len(final_df)}건의 데이터가 'combined_brand_reviews.csv'로 저장되었습니다.")
        
        # 그룹별 수집 결과 요약 출력
        print("\n--- 그룹별 수집 현황 ---")
        print(final_df['group'].value_counts())
    else:
        print("\n❌ 수집된 데이터가 하나도 없습니다.")

if __name__ == "__main__":
    main()


📂 [rextreme] 그룹 수집 시작 (목표: 500건, 정확도순)
📊 rextreme 진행도: 136/500 건 완료
📂 [brand] 그룹 수집 시작 (목표: 500건, 정확도순)
📊 brand 진행도: 500/500 건 완료

✨ 수집 완료! 최종 636건의 데이터가 'combined_brand_reviews.csv'로 저장되었습니다.

--- 그룹별 수집 현황 ---
group
brand       500
rextreme    136
Name: count, dtype: int64
